# M3L4 E11 — Golden Dataset sobre LangGraph + scores en Langfuse [OK] Resolution
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

## ¿Por qué importa este ejercicio?

En E04 armamos un golden dataset y medimos accuracy manualmente. En E08-E10 integramos Langfuse para tracing. **Ahora vamos a combinar ambos**: ejecutar el golden dataset contra LangGraph y registrar los scores automáticamente en Langfuse.

Esto nos da:
- **Trazabilidad completa**: cada caso del dataset tiene su trace en Langfuse
- **Scores automáticos**: el accuracy se registra como score en cada trace
- **Dashboard en Langfuse**: podés ver accuracy, errores y métricas sin salir de la plataforma

| Concepto | Definición simple | Cómo aparece acá |
|---|---|---|
| **Langfuse Score** | Calificación numérica asociada a un trace | `langfuse.create_score(trace_id, name='routing_correct', value=1/0)` |
| **Golden run** | Ejecución completa del golden dataset | Loop sobre `golden_dataset` invocando el grafo |
| **Evaluación automática** | Comparación expected vs actual sin intervención humana | `correct = int(actual == expected)` |

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph pandas
print('OK.')

In [ ]:
import os, pandas as pd
from getpass import getpass
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse import get_client
from langfuse.langchain import CallbackHandler

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('OK.')

In [ ]:
def route_query_v2(query):
    q = query.lower()
    det = []
    if any(w in q for w in ['vacaciones','licencia','recibo','nómina','rrhh']): det.append('hr')
    if any(w in q for w in ['vpn','error','app','laptop','wifi','login','contraseña']): det.append('it')
    if any(w in q for w in ['factura','pago','reembolso','gasto','cobro','comprobante','salario']): det.append('finance')
    if any(w in q for w in ['contrato','legal','confidencialidad','nda','acuerdo']): det.append('legal')
    if len(det) > 1: return 'multi_intent'
    if len(det) == 1: return det[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

class AgentState(TypedDict):
    query: str
    intent: str
    response: str

def router_node(s): return {'intent': route_query_v2(s['query'])}
def hr_node(s):      return {'response': 'HRAgent: Para vacaciones o recibos, ingresá al portal de RRHH.'}
def it_node(s):      return {'response': 'ITAgent: Para VPN o errores técnicos, abrí un ticket.'}
def finance_node(s): return {'response': 'FinanceAgent: Para facturas, revisá el portal de facturación.'}
def legal_node(s):   return {'response': 'LegalAgent: Para contratos o NDAs, contactá al equipo legal.'}
def general_node(s): return {'response': 'GeneralAgent: Necesito más información.'}
def route_to_node(s): return {'hr':'hr_node','it':'it_node','finance':'finance_node','legal':'legal_node'}.get(s['intent'],'general_node')

builder = StateGraph(AgentState)
builder.add_node('router_node', router_node)
for name, fn in [('hr_node',hr_node),('it_node',it_node),('finance_node',finance_node),('legal_node',legal_node),('general_node',general_node)]:
    builder.add_node(name, fn)
builder.set_entry_point('router_node')
builder.add_conditional_edges('router_node', route_to_node,
    {'hr_node':'hr_node','it_node':'it_node','finance_node':'finance_node','legal_node':'legal_node','general_node':'general_node'})
for node in ['hr_node','it_node','finance_node','legal_node','general_node']:
    builder.add_edge(node, END)
graph = builder.compile()
print('Grafo listo.')

In [ ]:
golden_dataset = [
    {'id': 'case_001', 'query': '¿Cómo solicito mis días de vacaciones?',          'expected_intent': 'hr'},
    {'id': 'case_002', 'query': 'Mi VPN no conecta desde ayer',                     'expected_intent': 'it'},
    {'id': 'case_003', 'query': 'Necesito ver mi factura del mes pasado',            'expected_intent': 'finance'},
    {'id': 'case_004', 'query': 'Necesito el contrato de confidencialidad actualizado', 'expected_intent': 'legal'},
    {'id': 'case_005', 'query': 'No puedo entrar al portal para ver mi recibo',     'expected_intent': 'hr'},
    {'id': 'case_006', 'query': 'ayuda',                                             'expected_intent': 'clarification'},
    {'id': 'case_007', 'query': '¿Cuándo se procesa el reembolso de gastos?',       'expected_intent': 'finance'},
    {'id': 'case_008', 'query': 'El sistema de login no me deja entrar',             'expected_intent': 'it'},
]
print(f'{len(golden_dataset)} casos.')

## Solución — Golden run con scores

Ejecutamos cada caso del golden dataset contra el grafo y:
1. Creamos un trace en Langfuse con `CallbackHandler`
2. Comparamos el intent detectado vs el esperado
3. Registramos un **score** en Langfuse usando `create_score()`
4. Almacenamos los resultados en un DataFrame

In [ ]:
langfuse = get_client()
results = []

for case in golden_dataset:
    langfuse_handler = CallbackHandler()

    output = graph.invoke(
        {'query': case['query'], 'intent': '', 'response': ''},
        config={
            'callbacks': [langfuse_handler],
            'metadata': {
                'langfuse_tags': ['golden-run-v1', 'm3l4'],
                'expected_intent': case['expected_intent']
            }
        }
    )

    actual_intent = output.get('intent', '')
    correct = int(actual_intent == case['expected_intent'])
    trace_id = getattr(langfuse_handler, 'last_trace_id', None)

    if trace_id:
        try:
            langfuse.create_score(
                trace_id=trace_id,
                name='routing_correct',
                value=correct,
                data_type='NUMERIC',
                comment=f"Expected {case['expected_intent']}, got {actual_intent}"
            )
        except Exception as e:
            print(f'  Warning scoring: {e}')

    results.append({
        'id': case['id'],
        'query': case['query'][:40],
        'expected_intent': case['expected_intent'],
        'actual_intent': actual_intent,
        'correct': correct,
        'trace_id': trace_id
    })
    print(f"{case['id']}: {case['expected_intent']} -> {actual_intent} {'[OK]' if correct else '[X]'}")

In [ ]:
df = pd.DataFrame(results)
print(f'Routing accuracy: {df["correct"].mean():.2%}')
df

## Verificación

In [ ]:
assert len(results) > 0
assert 'correct' in results[0]
accuracy = sum(r['correct'] for r in results) / len(results)
print(f'Checks E11 OK — Routing accuracy: {accuracy:.2%}')

## [OK] Cierre — ¿Qué logramos?

| Capacidad | Antes | Ahora |
|---|---|---|
| **Trazabilidad** | Métricas sueltas en pandas | Cada caso tiene su trace en Langfuse |
| **Scores** | Cálculo manual en DataFrame | Scores automáticos vinculados a cada trace |
| **Dashboard** | Notebook local | Podés ver todo en cloud.langfuse.com |

> **¿Por qué es importante?** Ahora cada ejecución del golden dataset deja un rastro completo en Langfuse. Podés volver a cualquier caso, ver el trace, el score, y entender exactamente qué pasó.

**¿Qué sigue?** En E12 cerramos el módulo con el **ciclo completo de mejora iterativa**: medir, diagnosticar, fixear y re-medir.